# Сборка датасета для модели сутяжности клиента

Пайплайн вынесен в пакет `querulus.dataset`. Тетрадка только настраивает окружение и вызывает `run_pipeline`.

## Настройка

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# Корень репозитория и src/ — как в integration/tests/__init__.py
_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
import logging

import pandas as pd

from querulus.dataset import cleanup_legacy_artifacts, run_pipeline
from querulus.dataset.io import setup_notebook_logging
from querulus.dataset.paths import DataPaths

setup_notebook_logging(logging.INFO)

# True: подгрузить готовый финальный датасет; False: пересобрать пайплайн
LOAD_FINAL_DATASET = True
FINAL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "df_final_3.parquet"

# False: parquet из data/raw (или SQL при отсутствии); True: всегда SQL
USE_SQL = False
SAVE_CHECKPOINT = True
# False: victim → targets; True: legacy enrich (не для обучения)
INCLUDE_ENRICH = False
# Пропустить victim/targets, продолжить с df_after_targets.parquet
RESUME_FROM_TARGETS = False
# Удалить legacy enrich / дубли raw перед сборкой (рекомендуется при падениях/OOM)
CLEANUP_LEGACY_ARTIFACTS = True
# Вместе с CLEANUP: сбросить processed-чекпоинты при полной пересборке
CLEANUP_STALE_PROCESSED = False

# Таргеты моделей (колонки датасета после build_targets):
#   frequency (классификация): "TARGET" | "TARGET_FREQ"
#   severity (регрессия):      "TARGET_SEV" | "TARGET_SEV_OLD"
FREQUENCY_TARGET = "TARGET_FREQ"
SEVERITY_TARGET = "TARGET_SEV"

if CLEANUP_LEGACY_ARTIFACTS and not LOAD_FINAL_DATASET:
    cleanup_legacy_artifacts(
        DataPaths.from_config(),
        enrich_only=True,
        redundant_raw=True,
        stale_processed=CLEANUP_STALE_PROCESSED,
    )

## Пайплайн

In [ ]:
if LOAD_FINAL_DATASET:
    if not FINAL_DATASET_PATH.exists():
        raise FileNotFoundError(
            f"Финальный датасет не найден: {FINAL_DATASET_PATH}. "
            "Поставьте LOAD_FINAL_DATASET=False, чтобы собрать его заново."
        )
    df = pd.read_parquet(FINAL_DATASET_PATH)
    print(f"LOAD final dataset: {FINAL_DATASET_PATH} shape={df.shape}")
else:
    df = run_pipeline(
        use_sql=USE_SQL,
        save_checkpoint=SAVE_CHECKPOINT,
        include_enrich=INCLUDE_ENRICH,
        resume_from_targets=RESUME_FROM_TARGETS,
    )

## Проверка результата

In [ ]:
for target_col in (FREQUENCY_TARGET, SEVERITY_TARGET):
    if target_col not in df.columns:
        raise KeyError(
            f"Нет колонки {target_col!r}. Пересоберите датасет: LOAD_FINAL_DATASET=False."
        )

print(f"frequency target: {FREQUENCY_TARGET}")
display(df[FREQUENCY_TARGET].value_counts(dropna=False))
print(f"severity target: {SEVERITY_TARGET}")
display(df[SEVERITY_TARGET].describe())

In [ ]:
preview_cols = [FREQUENCY_TARGET, SEVERITY_TARGET, "TARGET", "TARGET_SEV"]
if "TARGET_SEV_OLD" in df.columns:
    preview_cols.append("TARGET_SEV_OLD")
df[preview_cols].head()

In [ ]:
# Проверка несогласованности: TARGET_2 == 0 при TARGET_3_SEV/TARGET_SEV > 0
sev_col = "TARGET_3_SEV" if "TARGET_3_SEV" in df.columns else "TARGET_SEV"

needed = [
    "Сумма_выплат_по_претензиям",
    "Сумма_взыскано_по_ФУ",
    "Суммы_взыскано_по_иску",
]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Нет колонок для TARGET_2: {missing}")

target_2 = (
    df["Сумма_выплат_по_претензиям"].fillna(0)
    + df["Сумма_взыскано_по_ФУ"].fillna(0)
    + df["Суммы_взыскано_по_иску"].fillna(0)
)
mask = (target_2 == 0) & (df[sev_col].fillna(0) > 0)

print(f"Rows: {int(mask.sum())} / {len(df)}")
if "INCIDENT_NUMBER" in df.columns:
    print(f"Incidents: {df.loc[mask, 'INCIDENT_NUMBER'].nunique()}")

cols = [c for c in ["INCIDENT_NUMBER", "LOSS_NUMBER", sev_col] + needed if c in df.columns]
display(df.loc[mask, cols].head(50))

In [ ]:
# Сверка: TARGET_2 (ПСР) vs исковая логика (TARGET_FREQ / TARGET_SEV)
# TARGET_2: выплаты по претензиям + взыскано по ФУ + взыскано по судам (из ПСР)
# TARGET_FREQ_AMOUNT: RecoveredValueWithSD (последняя инстанция иска) + претензии *_all
# TARGET_SEV: RecoveredMainDebt/Wearout/LossCommody (последняя инстанция) + претензии *_all

import numpy as np
import pandas as pd

incident_col = "INCIDENT_NUMBER"
if incident_col not in df.columns:
    raise KeyError(f"Нет колонки {incident_col!r}")

# TARGET_2 amount
needed_t2 = [
    "Сумма_выплат_по_претензиям",
    "Сумма_взыскано_по_ФУ",
    "Суммы_взыскано_по_иску",
]
missing_t2 = [c for c in needed_t2 if c not in df.columns]
if missing_t2:
    raise KeyError(f"Нет колонок для TARGET_2: {missing_t2}")

t2_amount_row = (
    df[needed_t2[0]].fillna(0)
    + df[needed_t2[1]].fillna(0)
    + df[needed_t2[2]].fillna(0)
)

# Исковая логика (как в build_targets после пересборки)
if "TARGET_FREQ_AMOUNT" in df.columns:
    t3_component_row = df["TARGET_FREQ_AMOUNT"].fillna(0)
    t3_label = "TARGET_FREQ_AMOUNT"
elif {"TARGET_FREQ_CLAIMS_AMOUNT", "TARGET_FREQ_PRET_AMOUNT"}.issubset(df.columns):
    t3_component_row = (
        df["TARGET_FREQ_CLAIMS_AMOUNT"].fillna(0) + df["TARGET_FREQ_PRET_AMOUNT"].fillna(0)
    )
    t3_label = "TARGET_FREQ_CLAIMS_AMOUNT + TARGET_FREQ_PRET_AMOUNT"
else:
    raise KeyError(
        "Нет TARGET_FREQ_AMOUNT (или TARGET_FREQ_CLAIMS_AMOUNT + TARGET_FREQ_PRET_AMOUNT). "
        "Пересоберите датасет: LOAD_FINAL_DATASET=False."
    )

if "TARGET_SEV" in df.columns:
    print(
        "TARGET_SEV sample:",
        df[["TARGET_SEV", "TARGET_SEV_CLAIMS_AMOUNT"] if "TARGET_SEV_CLAIMS_AMOUNT" in df.columns else ["TARGET_SEV"]]
        .head(3)
        .to_string(index=False),
    )
print(f"t3_component source: {t3_label}")

# Сверяем на уровне инцидента (на всякий случай, если в df есть дубли внутри инцидента)
check = (
    pd.DataFrame(
        {
            incident_col: df[incident_col],
            "t2_amount": t2_amount_row,
            "t3_component": t3_component_row,
        }
    )
    .groupby(incident_col, as_index=False)
    .sum()
)

check["abs_diff"] = (check["t2_amount"] - check["t3_component"]).abs()
# % относительно TARGET_2 (если TARGET_2==0, берём 100% только если t3_component>0)
check["pct_diff_vs_t2"] = np.where(
    check["t2_amount"] > 0,
    check["abs_diff"] / check["t2_amount"],
    np.where(check["t3_component"] > 0, 1.0, 0.0),
)

# Порог "сходится" (1%)
match_threshold = 0.01
check["is_match"] = check["pct_diff_vs_t2"] <= match_threshold

print(f"Incidents total: {len(check)}")
print(f"Match (<= {match_threshold:.0%}): {int(check['is_match'].sum())} ({check['is_match'].mean():.1%})")
print(f"Median pct diff: {check['pct_diff_vs_t2'].median():.2%}")

# Топ расхождений
out = check.sort_values(["pct_diff_vs_t2", "abs_diff"], ascending=False).head(50)
out["pct_diff_vs_t2"] = (out["pct_diff_vs_t2"] * 100).round(2)
display(out[[incident_col, "t2_amount", "t3_component", "abs_diff", "pct_diff_vs_t2"]])

## EDA признаков

`research_continous` / `research_feature` по MVP-признакам (до обучения, как в `model_learn.py`).

In [ ]:
from querulus.training.config import TrainingConfig
from querulus.training.plots import run_mvp_frequency_eda

EDA_CONFIG = TrainingConfig(
    frequency_target=FREQUENCY_TARGET,
    severity_target=SEVERITY_TARGET,
)
if "expos" not in df.columns:
    df["expos"] = 1

run_mvp_frequency_eda(df, EDA_CONFIG)

## Обучение моделей

Обучение вынесено в отдельный пакет `querulus.training`. Сборка датасета и обучение запускаются разными ячейками.

In [ ]:
import importlib

importlib.invalidate_caches()

from IPython.display import Markdown, display

from querulus.training.config import TrainingConfig
from querulus.training.pipeline import (
    format_features_table,
    format_metrics_table,
    format_training_summary,
    train_models,
)
# None = полный MVP-пул; frequency: + select_features при frequency_select_features=True.
FREQUENCY_FEATURES = None
SEVERITY_FEATURES = None

TRAINING_CONFIG = TrainingConfig(
    frequency_target=FREQUENCY_TARGET,
    severity_target=SEVERITY_TARGET,
    frequency_features=FREQUENCY_FEATURES,
    severity_features=SEVERITY_FEATURES,
    frequency_select_features=False,
    frequency_num_features_to_select=20,
    frequency_calibration_enabled=False,
)
training = train_models(df, TRAINING_CONFIG)

display(Markdown("### Training summary"))
display(format_training_summary(training.summary))

display(Markdown("### Frequency features"))
display(format_features_table(training.frequency_features, training.frequency_categorical_features))

display(Markdown("### Severity features"))
display(format_features_table(training.severity_features, training.severity_categorical_features))

display(Markdown("### Frequency (classification)"))
display(format_metrics_table(training.frequency_metrics_table))
display(Markdown("### Severity (regression)"))
display(format_metrics_table(training.severity_metrics_table))

display(Markdown("### Frequency importance"))
display(training.frequency_importance.head(30))
display(Markdown("### Severity importance"))
display(training.severity_importance.head(30))

## Диагностика модели

Распределение вероятностей классификации и ModelDiagnostics (после обучения).

In [ ]:
from querulus.training.plots import run_model_diagnostics_visualizations

run_model_diagnostics_visualizations(df, training, TRAINING_CONFIG)

## Финансовый эффект

Расчёт факта/модели, подбор порога классификации, сводная таблица и экспорт — пакет `querulus.fin_effect` (логика из Litigant `fin_effect.py`).

In [ ]:
from querulus.fin_effect import (
    FinEffectConfig,
    create_summary_table,
    export_analytics_excel,
    export_summary_excel,
    print_best_threshold_report,
    run_fin_effect_from_training,
)

FIN_EFFECT_CONFIG = FinEffectConfig(
    frequency_target_column=FREQUENCY_TARGET,
    severity_target_column=SEVERITY_TARGET,
    threshold_step=0.01,
)
FIN_EFFECT_DIR = PROJECT_ROOT / "data" / "processed" / "fin_effect"
FIN_EFFECT_DIR.mkdir(parents=True, exist_ok=True)

# test-сплит по дате (как в обучении); порог — max net_effect (шаг 0.01)
fin_effect = run_fin_effect_from_training(
    df, training, split="test", config=FIN_EFFECT_CONFIG
)
print_best_threshold_report(fin_effect)

summary_table = create_summary_table(fin_effect.frame, FIN_EFFECT_CONFIG)
display(summary_table.style.format("{:,.0f}", subset=summary_table.columns[3:]))

export_summary_excel(summary_table, FIN_EFFECT_DIR / "summary_table.xlsx")
export_analytics_excel(
    fin_effect.frame, FIN_EFFECT_DIR / "Аналитика.xlsx", config=FIN_EFFECT_CONFIG
)
print(f"Экспорт: {FIN_EFFECT_DIR}")

In [ ]:
from querulus.fin_effect import (
    plot_confusion_matrix,
    plot_cost_confusion_heatmaps,
    plot_positive_cases_by_month,
    plot_precision_recall_vs_threshold,
    plot_severity_fact_vs_pred_binned,
    plot_target_monthly_share,
)

split = training.frequency_split
frame = fin_effect.frame
y_test = split.y_test.loc[frame.index]
x_test = split.x_test.loc[frame.index, training.frequency_features]

business_threshold = fin_effect.best_threshold
plot_confusion_matrix(
    frame[FIN_EFFECT_CONFIG.frequency_target_column],
    frame["pred_freq"],
    title=f"Confusion Matrix {FREQUENCY_TARGET} (threshold={business_threshold:.2f})",
)
plot_precision_recall_vs_threshold(training.frequency_model, x_test, y_test)
plot_cost_confusion_heatmaps(frame, y_test, frame["pred_freq"])
plot_severity_fact_vs_pred_binned(
    frame,
    frame[SEVERITY_TARGET],
    frame["pred_sev"],
    mask_actual_claim=(y_test == 1),
    config=FIN_EFFECT_CONFIG,
)
plot_target_monthly_share(df, config=FIN_EFFECT_CONFIG)
plot_positive_cases_by_month(df, config=FIN_EFFECT_CONFIG)